# Кейс 3. Кластеризация клиентов банка в Python

Эта тетрадь показывает пошаговый сценарий работы с данными: загрузка CSV, отбор признаков, кластеризация `K-means`, визуализация в 2D/3D и формирование списка клиентов для коммуникации.

## Что мы хотим доказать

Простая сегментация по одному признаку не подходит. В данных есть клиенты с высоким оборотом и хорошей платежной дисциплиной, но со слабой маржой. Чтобы увидеть такой паттерн, нам нужна именно многомерная кластеризация.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

In [ ]:
BASE_DIR = None
for candidate in [Path.cwd(), Path.cwd() / "python_client_clusters", Path.cwd().parent / "python_client_clusters"]:
    if (candidate / "data" / "bank_client_behavior.csv").exists():
        BASE_DIR = candidate
        break

if BASE_DIR is None:
    raise FileNotFoundError("Не найден каталог с data/bank_client_behavior.csv. Откройте папку python_client_clusters в IDE или запустите open_notebook.sh.")

DATA_PATH = BASE_DIR / "data" / "bank_client_behavior.csv"
LEADS_PATH = BASE_DIR / "data" / "target_clients_from_notebook.csv"

df = pd.read_csv(DATA_PATH)
print(f"Файл: {DATA_PATH}")
print(f"Строк: {len(df):,}")
print(f"Колонок: {df.shape[1]}")

In [ ]:
display(df.head())

## Выбираем признаки

Берем не только оборот, но и поведенческие характеристики: использование лимита, долю полной оплаты, просрочки, снятие наличных, вовлеченность в цифровые каналы, депозитный баланс и итоговую маржу.

In [ ]:
feature_cols = [
    "avg_monthly_spend_rub",
    "utilization_pct",
    "full_pay_ratio_pct",
    "overdue_events_6m",
    "cash_advance_share_pct",
    "installment_share_pct",
    "digital_engagement_index",
    "deposit_balance_rub",
    "travel_spend_share_pct",
    "total_margin_rub",
]

display(df[feature_cols].describe().T.round(2))

## Нормализация и выбор числа кластеров

Чтобы признаки с большими шкалами не доминировали, стандартизируем их. Затем смотрим на `silhouette score` и инерцию для нескольких значений `k`.

In [ ]:
scaler = StandardScaler()
scaled = scaler.fit_transform(df[feature_cols])

scores = []
for k in range(3, 8):
    model = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = model.fit_predict(scaled)
    scores.append({
        "k": k,
        "silhouette": silhouette_score(scaled, labels),
        "inertia": model.inertia_,
    })

scores_df = pd.DataFrame(scores)
display(scores_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scores_df["k"], scores_df["silhouette"], marker="o", color="#21A038")
axes[0].set_title("Silhouette score по числу кластеров")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette score")

axes[1].plot(scores_df["k"], scores_df["inertia"], marker="o", color="#0B1F33")
axes[1].set_title("Инерция по числу кластеров")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Inertia")
plt.tight_layout()

Для этого кейса берем `k = 5`: это дает устойчивое разделение по профилям и хорошо читается на занятии.

In [ ]:
model = KMeans(n_clusters=5, n_init=20, random_state=42)
df["kmeans_cluster"] = model.fit_predict(scaled)
df["cluster_label"] = df["kmeans_cluster"].map(lambda x: f"Кластер {x}")

df[["client_id", "cluster_label"]].head()

## Проекция в двумерное пространство

Снижаем размерность через PCA, чтобы показать кластеры на плоскости.

In [ ]:
pca = PCA(n_components=3, random_state=42)
components = pca.fit_transform(scaled)
df["pc1"] = components[:, 0]
df["pc2"] = components[:, 1]
df["pc3"] = components[:, 2]

colors = ["#21A038", "#4F88FF", "#FCCB00", "#9B59B6", "#E67E22"]

fig, ax = plt.subplots(figsize=(11, 6))
for cluster_id in sorted(df["kmeans_cluster"].unique()):
    subset = df[df["kmeans_cluster"] == cluster_id]
    ax.scatter(
        subset["pc1"],
        subset["pc2"],
        s=22,
        alpha=0.7,
        label=f"Кластер {cluster_id}",
        color=colors[cluster_id],
    )

ax.set_title("Кластеры K-means в двумерной проекции")
ax.set_xlabel("Главная компонента 1")
ax.set_ylabel("Главная компонента 2")
ax.legend(frameon=False, ncol=3)
plt.tight_layout()

## Проекция в 3D

Здесь особенно хорошо видно, что похожие по обороту клиенты расходятся по другим факторам и не укладываются в простую сегментацию по диапазонам.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

for cluster_id in sorted(df["kmeans_cluster"].unique()):
    subset = df[df["kmeans_cluster"] == cluster_id]
    ax.scatter(
        subset["pc1"],
        subset["pc2"],
        subset["pc3"],
        s=18,
        alpha=0.7,
        label=f"Кластер {cluster_id}",
        color=colors[cluster_id],
    )

ax.set_title("Почему простых порогов недостаточно: 3D-проекция")
ax.set_xlabel("ГК1")
ax.set_ylabel("ГК2")
ax.set_zlabel("ГК3")
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()

## Портреты кластеров

Теперь агрегируем ключевые показатели по кластерам и переводим математику на язык бизнеса.

In [ ]:
cluster_profile = (
    df.groupby("kmeans_cluster")
    .agg(
        clients=("client_id", "count"),
        avg_monthly_spend_rub=("avg_monthly_spend_rub", "mean"),
        utilization_pct=("utilization_pct", "mean"),
        full_pay_ratio_pct=("full_pay_ratio_pct", "mean"),
        overdue_events_6m=("overdue_events_6m", "mean"),
        cash_advance_share_pct=("cash_advance_share_pct", "mean"),
        installment_share_pct=("installment_share_pct", "mean"),
        deposit_balance_rub=("deposit_balance_rub", "mean"),
        total_margin_rub=("total_margin_rub", "mean"),
    )
    .round(2)
    .reset_index()
)

display(cluster_profile)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(cluster_profile["kmeans_cluster"].astype(str), cluster_profile["total_margin_rub"], color="#21A038")
ax2 = ax.twinx()
ax2.plot(cluster_profile["kmeans_cluster"].astype(str), cluster_profile["avg_monthly_spend_rub"], color="#FCCB00", marker="o", linewidth=2.2)
ax.set_title("Портреты кластеров: высокий оборот не гарантирует маржу")
ax.set_ylabel("Средняя маржа, руб.")
ax2.set_ylabel("Средний оборот, руб.")
plt.tight_layout()

## Выбираем кластер для коммуникации

Ищем кластер, где одновременно:

- высокий средний оборот;
- высокая доля полной оплаты;
- низкая итоговая маржа.

Для него логичнее не продвигать кредит, а тестировать премиальный или инвестиционный оффер.

In [ ]:
target_cluster = (
    cluster_profile.sort_values(
        ["avg_monthly_spend_rub", "full_pay_ratio_pct", "total_margin_rub"],
        ascending=[False, False, True],
    )
    .iloc[0]["kmeans_cluster"]
)

print(f"Целевой кластер: {int(target_cluster)}")
display(cluster_profile[cluster_profile["kmeans_cluster"] == target_cluster])

In [ ]:
df["recommended_action"] = "Без кампании на этом этапе"
df.loc[df["kmeans_cluster"] == target_cluster, "recommended_action"] = "Предложить премиальный travel-пакет или инвестиционный оффер"

leads = (
    df[df["kmeans_cluster"] == target_cluster]
    .sort_values(["avg_monthly_spend_rub", "deposit_balance_rub"], ascending=False)
    [[
        "client_id",
        "avg_monthly_spend_rub",
        "deposit_balance_rub",
        "full_pay_ratio_pct",
        "total_margin_rub",
        "recommended_action",
    ]]
    .head(30)
)

leads.to_csv(LEADS_PATH, index=False)
print(f"Список клиентов сохранен в: {LEADS_PATH}")
display(leads.head(10))

## Финальный вывод для аудитории

1. Без кластеризации этот кейс легко упростить до ошибочной сегментации по обороту.
2. `K-means` показывает, что часть "хороших" клиентов выглядит безопасно, но экономически недоиспользована.
3. После этого аналитика сразу превращается в действие: мы получаем конкретный список клиентов для тестовой коммуникации.